# Exploración de datos de Instacart con Python y Pandas

**Aprendiz:** Juan Camilo Montes Velásquez  
**Programa:** Tecnólogo en Análisis y Desarrollo de Software, SENA  
**Evidencia:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas  

*Nota: los cinco archivos CSV deben estar en la carpeta `data/raw/` del repositorio (ver README).*

## Momento 1. Comprensión del problema y reconocimiento de los datos

### Descripción del problema
Instacart es una plataforma de compras de supermercado. El equipo de análisis necesita saber qué información guardan sus archivos de pedidos, productos, departamentos y pasillos, y cómo consultarla para entender cómo compran los clientes. Mi papel es el de analista de datos: hacer una primera exploración con Python y Pandas, ver la estructura de los datos, consultar registros específicos y sacar indicadores básicos.

### Los cinco archivos y qué guarda cada uno

| Archivo | Qué información almacena |
|---|---|
| `instacart_orders.csv` | Un registro por pedido: el usuario que lo hizo (`user_id`), el número de pedido de ese usuario (`order_number`), el día de la semana (`order_dow`), la hora (`order_hour_of_day`) y los días desde su pedido anterior (`days_since_prior_order`). |
| `products.csv` | El catálogo: cada producto con su nombre, su pasillo (`aisle_id`) y su departamento (`department_id`). |
| `aisles.csv` | El nombre de cada pasillo, identificado por `aisle_id`. |
| `departments.csv` | El nombre de cada departamento, identificado por `department_id`. |
| `order_products.csv` | Qué productos van en cada pedido, en qué posición se agregaron al carrito (`add_to_cart_order`) y si fue una recompra (`reordered`). |

### Columnas para identificar pedidos, usuarios y productos
- **Pedidos:** `order_id` (en `instacart_orders` y `order_products`).
- **Usuarios:** `user_id` (solo en `instacart_orders`).
- **Productos:** `product_id` (en `products` y `order_products`).

Así se relacionan los archivos: `order_products.order_id` con `instacart_orders.order_id`; `order_products.product_id` con `products.product_id`; `products.aisle_id` con `aisles.aisle_id`; y `products.department_id` con `departments.department_id`.

### Tres preguntas de análisis
1. ¿En qué horas del día se concentran los pedidos?
2. ¿Cada cuánto tiempo suelen volver a comprar los clientes?
3. ¿Qué parte de los productos que se compran son recompras?

Las respondo al final, en el Momento 6.

### Por qué es importante explorar los datos antes de analizarlos
Antes de calcular algo hay que saber qué datos tengo: cuántos registros son, qué significa cada columna, si faltan valores y cómo se conectan los archivos. Si me salto este paso puedo interpretar mal los resultados (por ejemplo, confundir filas con clientes) o usar datos incompletos sin darme cuenta.

## Momento 2. Importación y exploración inicial de los archivos

In [1]:
import pandas as pd

# Los archivos usan punto y coma como separador, por eso sep=";"
# Ajustar las rutas según el entorno de trabajo
orders = pd.read_csv("../data/raw/instacart_orders.csv", sep=";")
products = pd.read_csv("../data/raw/products.csv", sep=";")
aisles = pd.read_csv("../data/raw/aisles.csv", sep=";")
departments = pd.read_csv("../data/raw/departments.csv", sep=";")
order_products = pd.read_csv("../data/raw/order_products.csv", sep=";")

# Los guardo en un diccionario para poder recorrerlos
dfs = {
    "instacart_orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

### Primeros y últimos cinco registros de cada DataFrame

In [2]:
for nombre, df in dfs.items():
    print("=" * 60)
    print(nombre)
    print("Primeros 5 registros:")
    display(df.head())
    print("Últimos 5 registros:")
    display(df.tail())

instacart_orders
Primeros 5 registros:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


Últimos 5 registros:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


products
Primeros 5 registros:


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


... (10 outputs left out)

order_products
Primeros 5 registros:


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


Últimos 5 registros:


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


### Dimensiones de cada DataFrame

In [3]:
for nombre, df in dfs.items():
    print(f"{nombre}: {df.shape[0]} filas y {df.shape[1]} columnas")

print()
print("Más registros:", max(dfs, key=lambda k: dfs[k].shape[0]))
menos = min(v.shape[1] for v in dfs.values())
print("Menos columnas:", [k for k, v in dfs.items() if v.shape[1] == menos])

instacart_orders: 478967 filas y 6 columnas
products: 49694 filas y 4 columnas
aisles: 134 filas y 2 columnas
departments: 21 filas y 2 columnas
order_products: 4545007 filas y 4 columnas

Más registros: order_products
Menos columnas: ['aisles', 'departments']


### Respuestas del Momento 2
- **¿Cuál contiene más registros?** `order_products`, con 4.545.007 filas.
- **¿Cuál contiene menos columnas?** `aisles` y `departments`, con 2 columnas cada uno.
- **Diferencias entre pedidos y productos:** `instacart_orders` describe *cuándo y quién* compra (usuario, día, hora, frecuencia) y tiene una fila por pedido. `products` describe *qué se vende* (nombre, pasillo, departamento) y es un catálogo que no cambia con cada pedido.
- **Por qué importa conocer las dimensiones:** me dicen qué tan grande es el conjunto (con 4,5 millones de filas, `order_products` usa más memoria y tarda más en procesarse), con cuántos datos estoy trabajando y sirven para detectar errores de carga. Por ejemplo, si importara sin `sep=";"` todo quedaría en una sola columna y el `shape` lo mostraría.

## Momento 3. Reconocimiento de la estructura y calidad inicial de los datos

In [4]:
for nombre, df in dfs.items():
    print("=" * 60)
    print(nombre)
    print("Columnas:", list(df.columns))
    print()
    print(df.dtypes)
    print()
    df.info(show_counts=True)

instacart_orders
Columnas: ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB
products
Columnas: ['product_id', 'product_name', 'aisle_id', 'departme

### Valores ausentes por columna

In [5]:
for nombre, df in dfs.items():
    nulos = df.isna().sum()
    print(nombre, "->", nulos[nulos > 0].to_dict() or "sin valores nulos")

instacart_orders -> {'days_since_prior_order': 28819}
products -> {'product_name': 1258}
aisles -> sin valores nulos
departments -> sin valores nulos
order_products -> {'add_to_cart_order': 836}


### Revisión de dónde salen los valores ausentes
Aquí solo miro los datos para entenderlos. No corrijo ni elimino nada en esta fase.

In [6]:
# ¿Qué pedidos no tienen days_since_prior_order?
sin_dias = orders[orders["days_since_prior_order"].isna()]
print("Pedidos sin days_since_prior_order:", len(sin_dias))
print("¿Todos son el primer pedido del usuario (order_number == 1)?", (sin_dias["order_number"] == 1).all())
print("Pedidos con order_number == 1 en total:", (orders["order_number"] == 1).sum())
print()

# ¿Qué productos no tienen nombre?
sin_nombre = products[products["product_name"].isna()]
print("Productos sin nombre:", len(sin_nombre))
print("department_id de esos productos:", sin_nombre["department_id"].unique())
print("aisle_id de esos productos:", sin_nombre["aisle_id"].unique())
print()

# ¿Hay pedidos repetidos?
print("Filas completamente duplicadas en instacart_orders:", orders.duplicated().sum())

Pedidos sin days_since_prior_order: 28819
¿Todos son el primer pedido del usuario (order_number == 1)? True
Pedidos con order_number == 1 en total: 28819

Productos sin nombre: 1258
department_id de esos productos: [21]
aisle_id de esos productos: [100]



Filas completamente duplicadas en instacart_orders: 15


### Tabla de diagnóstico

In [7]:
observaciones = {
    "instacart_orders": "28.819 nulos en days_since_prior_order (son los primeros pedidos). 15 filas duplicadas.",
    "products": "1.258 productos sin nombre; todos están en el departamento 21 y el pasillo 100, que se llaman 'missing'.",
    "aisles": "Sin nulos. El pasillo 100 se llama 'missing'.",
    "departments": "Sin nulos. El departamento 21 se llama 'missing'.",
    "order_products": "836 nulos en add_to_cart_order. Es el archivo más grande.",
}

diagnostico = pd.DataFrame({
    "archivo": list(dfs.keys()),
    "filas": [df.shape[0] for df in dfs.values()],
    "columnas": [df.shape[1] for df in dfs.values()],
    "valores_nulos": [int(df.isna().sum().sum()) for df in dfs.values()],
})
diagnostico["observaciones"] = diagnostico["archivo"].map(observaciones)

pd.set_option("display.max_colwidth", None)
diagnostico

,archivo,filas,columnas,valores_nulos,observaciones
0,instacart_orders,478967,6,28819,28.819 nulos en days_since_prior_order (son los primeros pedidos). 15 filas duplicadas.
1,products,49694,4,1258,"1.258 productos sin nombre; todos están en el departamento 21 y el pasillo 100, que se llaman 'missing'."
2,aisles,134,2,0,Sin nulos. El pasillo 100 se llama 'missing'.
3,departments,21,2,0,Sin nulos. El departamento 21 se llama 'missing'.
4,order_products,4545007,4,836,836 nulos en add_to_cart_order. Es el archivo más grande.


### Respuestas del Momento 3
- **Diferencias entre `int64`, `float64` y `object`:** `int64` guarda números enteros, `float64` guarda números con decimales y `object` guarda texto (en versiones nuevas de Pandas el texto puede aparecer como `str`). Algunas columnas que serían enteras, como `days_since_prior_order` y `add_to_cart_order`, aparecen como `float64` porque tienen valores ausentes.
- **Qué da `info()` que no da `shape`:** `shape` solo dice cuántas filas y columnas hay. `info()` además muestra el nombre y el tipo de cada columna, cuántos valores no nulos tiene y la memoria que ocupa.
- **Columnas con valores ausentes:** `days_since_prior_order` en `instacart_orders` (28.819), `product_name` en `products` (1.258) y `add_to_cart_order` en `order_products` (836). Los 28.819 nulos de `days_since_prior_order` son justo los pedidos con `order_number == 1`: el primer pedido de un usuario no tiene pedido anterior. Los productos sin nombre pertenecen todos al departamento y pasillo llamados `missing`.
- **Dificultades que pueden causar:** los promedios y conteos ignoran los nulos, así que el promedio de `days_since_prior_order` no incluye los primeros pedidos. Los filtros como `> 15` tampoco incluyen los nulos. Los productos sin nombre no se pueden analizar por nombre, y los nulos de `add_to_cart_order` afectan cualquier análisis sobre el orden en que se agregan los productos.
- **Otro detalle:** `instacart_orders` tiene 15 filas duplicadas, que habría que revisar antes de contar pedidos.

## Momento 4. Indexación y filtrado lógico de la información

### Ejercicio 1. Seleccionar `product_id` y `product_name` (products)

In [8]:
ej1 = products[["product_id", "product_name"]]
print(ej1.shape)
ej1.head()

(49694, 2)


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce
4,5,Green Chile Anytime Sauce


Se seleccionaron solo 2 columnas, pero se conservan las 49.694 filas del catálogo.

### Ejercicio 2. Productos con `department_id` igual a 1 (products)

In [9]:
ej2 = products.loc[products["department_id"] == 1]
print(ej2.shape)
ej2.head()

(4008, 4)


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


Hay 4.008 productos del departamento 1 (según `departments.csv`, es *frozen*). Aquí cambian las filas y se conservan las 4 columnas.

### Ejercicio 3. Pedidos realizados a las 10 horas (orders)

In [10]:
ej3 = orders.loc[orders["order_hour_of_day"] == 10]
print(ej3.shape)
ej3.head()

(40578, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


Se hicieron 40.578 pedidos a las 10 de la mañana.

### Ejercicio 4. Pedidos con `order_dow` igual a 0 (orders)

In [11]:
ej4 = orders.loc[orders["order_dow"] == 0]
print(ej4.shape)
ej4.head()

(84090, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


Hay 84.090 pedidos con `order_dow` igual a 0. El archivo no aclara a qué día de la semana corresponde el 0, así que no lo interpreto como un día en particular.

### Ejercicio 5. Pedidos con `order_number` superior a 10 (orders)

In [12]:
ej5 = orders.loc[orders["order_number"] > 10]
print(ej5.shape)
ej5.head()

(244072, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


Hay 244.072 pedidos que son el pedido número 11 o más de su usuario, es decir, de clientes que ya han comprado varias veces.

### Ejercicio 6. Pedidos con `days_since_prior_order` superior a 15 (orders)

In [13]:
ej6 = orders.loc[orders["days_since_prior_order"] > 15]
print(ej6.shape)
ej6.head()

(110706, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


Hay 110.706 pedidos que llegaron más de 15 días después del anterior. Los pedidos con valor nulo no entran en este filtro, porque comparar un nulo con `> 15` da falso.

### Ejercicio 7. Registros con `reordered` igual a 1 (order_products)

In [14]:
ej7 = order_products.loc[order_products["reordered"] == 1]
print(ej7.shape)
ej7.head()

(2683838, 4)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


Hay 2.683.838 artículos que son recompras.

### Ejercicio 8. Productos en la primera posición del carrito (order_products)

In [15]:
ej8 = order_products.loc[order_products["add_to_cart_order"] == 1]
print(ej8.shape)
ej8.head()

(450046, 4)


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


Hay 450.046 registros en la primera posición, que coincide con la cantidad de pedidos distintos que tiene `order_products` (se confirma en el Momento 5): cada pedido tiene un primer artículo.

### Filtro con dos condiciones A. Pedidos a las 10 horas y con `order_number` superior a 10

In [16]:
filtro_a = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
print(filtro_a.shape)
filtro_a.head()

(21537, 6)


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


De los 40.578 pedidos de las 10 horas, solo 21.537 son además de usuarios con más de 10 pedidos.

### Filtro con dos condiciones B. Productos del departamento 1 con `aisle_id` superior a 50

In [17]:
filtro_b = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]
print(filtro_b.shape)
filtro_b.head()

(1618, 4)


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


De los 4.008 productos del departamento 1, solo 1.618 están en un pasillo con `aisle_id` mayor a 50.

### Respuestas del Momento 4
- **Seleccionar columnas vs. filtrar filas:** seleccionar columnas deja todas las filas pero solo con las columnas que pido (ejercicio 1: 49.694 filas y 2 columnas). Filtrar filas deja todas las columnas pero solo con los registros que cumplen la condición (ejercicio 2: 4.008 filas y 4 columnas).
- **Ventajas de `loc[]`:** permite filtrar filas con una condición y, si quiero, elegir las columnas en la misma instrucción (por ejemplo `products.loc[products["department_id"] == 1, ["product_id", "product_name"]]`), y el código se lee de forma clara.
- **Qué pasa con la cantidad de registros al aplicar varias condiciones:** disminuye, porque cada condición con `&` exige algo más. Los pedidos a las 10 horas bajaron de 40.578 a 21.537, y los productos del departamento 1 bajaron de 4.008 a 1.618.
- **Por qué usar bien los operadores lógicos:** en Pandas se usan `&` (y), `|` (o) y `~` (no), con cada condición entre paréntesis. Si uso `and` o `or` da error, y si cambio `&` por `|` el resultado cambia por completo (en vez de reducir las filas las amplía).

## Momento 5. Obtención e interpretación de indicadores básicos

In [18]:
n_pedidos = orders.shape[0]
n_productos = products.shape[0]
prom_order_number = orders["order_number"].mean()
prom_dias = orders["days_since_prior_order"].mean()
n_reordered = (order_products["reordered"] == 1).sum()
suma_reordered = order_products["reordered"].sum()
desc_hora = orders["order_hour_of_day"].describe()
desc_carrito = order_products["add_to_cart_order"].describe()

print("Registros de pedidos:", n_pedidos)
print("Registros de productos:", n_productos)
print("Promedio de order_number:", prom_order_number)
print("Promedio de days_since_prior_order:", prom_dias)
print("Registros con reordered == 1:", n_reordered)
print("Suma de reordered:", suma_reordered)
print("Valores que toma reordered:", order_products["reordered"].unique())
print()
print("describe() de order_hour_of_day")
print(desc_hora)
print()
print("describe() de add_to_cart_order")
print(desc_carrito)

Registros de pedidos: 478967
Registros de productos: 49694
Promedio de order_number: 17.157808366755955
Promedio de days_since_prior_order: 11.101750979677794
Registros con reordered == 1: 2683838
Suma de reordered: 2683838
Valores que toma reordered: [0 1]

describe() de order_hour_of_day
count    478967.000000
mean         13.446676
std           4.224986
min           0.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          23.000000
Name: order_hour_of_day, dtype: float64

describe() de add_to_cart_order
count    4.544171e+06
mean     8.351669e+00
std      7.080368e+00
min      1.000000e+00
25%      3.000000e+00
50%      6.000000e+00
75%      1.100000e+01
max      6.400000e+01
Name: add_to_cart_order, dtype: float64


### Tabla de resultados

In [19]:
def miles(n):
    return f"{int(n):,}".replace(",", ".")

def dec(x):
    return f"{x:.2f}".replace(".", ",")

resultados = pd.DataFrame([
    ["¿Cuántos registros de pedidos contiene el archivo?", miles(n_pedidos),
     "Cada fila es un registro de pedido. No es el número de clientes (ver más abajo)."],
    ["¿Cuántos registros de productos contiene el catálogo?", miles(n_productos),
     "El catálogo tiene 49.694 productos."],
    ["¿Cuál es el promedio de order_number?", dec(prom_order_number),
     "En promedio, un pedido es el número 17 de su usuario, lo que indica que hay clientes con historial largo."],
    ["¿Cuál es el promedio de days_since_prior_order?", dec(prom_dias),
     "En promedio pasan unos 11 días entre un pedido y el anterior (sin contar los primeros pedidos, que no tienen dato)."],
    ["¿Cuántos registros de artículos tienen reordered igual a 1?", miles(n_reordered),
     "Más de 2,6 millones de artículos comprados fueron recompras."],
    ["¿Cuál es la suma de reordered?", miles(suma_reordered),
     "Es igual al conteo anterior porque reordered solo vale 0 o 1: sumarla equivale a contar los 1."],
    ["¿Qué características estadísticas presenta order_hour_of_day?",
     f"media {dec(desc_hora['mean'])}; mediana {int(desc_hora['50%'])}; mín {int(desc_hora['min'])}; máx {int(desc_hora['max'])}",
     "La mitad de los pedidos se hace entre las 10 y las 16 horas (cuartiles 25 % y 75 %) y la mediana es las 13 horas."],
    ["¿Qué características estadísticas presenta add_to_cart_order?",
     f"media {dec(desc_carrito['mean'])}; mediana {int(desc_carrito['50%'])}; mín {int(desc_carrito['min'])}; máx {int(desc_carrito['max'])}",
     "La mayoría de los pedidos tiene pocos artículos (mediana 6), pero la media es mayor (8,35) y el máximo llega a 64, porque hay pedidos muy grandes. Tiene 836 valores menos que los registros por los nulos."],
], columns=["Pregunta", "Valor obtenido", "Interpretación"])

resultados

,Pregunta,Valor obtenido,Interpretación
0,¿Cuántos registros de pedidos contiene el archivo?,478.967,Cada fila es un registro de pedido. No es el número de clientes (ver más abajo).
1,¿Cuántos registros de productos contiene el catálogo?,49.694,El catálogo tiene 49.694 productos.
2,¿Cuál es el promedio de order_number?,"17,16","En promedio, un pedido es el número 17 de su usuario, lo que indica que hay clientes con historial largo."
3,¿Cuál es el promedio de days_since_prior_order?,"11,10","En promedio pasan unos 11 días entre un pedido y el anterior (sin contar los primeros pedidos, que no tienen dato)."
4,¿Cuántos registros de artículos tienen reordered igual a 1?,2.683.838,"Más de 2,6 millones de artículos comprados fueron recompras."
5,¿Cuál es la suma de reordered?,2.683.838,Es igual al conteo anterior porque reordered solo vale 0 o 1: sumarla equivale a contar los 1.
6,¿Qué características estadísticas presenta order_hour_of_day?,"media 13,45; mediana 13; mín 0; máx 23",La mitad de los pedidos se hace entre las 10 y las 16 horas (cuartiles 25 % y 75 %) y la mediana es las 13 horas.
7,¿Qué características estadísticas presenta add_to_cart_order?,"media 8,35; mediana 6; mín 1; máx 64","La mayoría de los pedidos tiene pocos artículos (mediana 6), pero la media es mayor (8,35) y el máximo llega a 64, porque hay pedidos muy grandes. Tiene 836 valores menos que los registros por los nulos."


### Registros, pedidos y usuarios no son lo mismo
Una fila no es un cliente. Aquí lo compruebo:

In [20]:
print("Registros en instacart_orders:", len(orders))
print("Pedidos distintos (order_id):", orders["order_id"].nunique())
print("Usuarios distintos (user_id):", orders["user_id"].nunique())
print()
print("Registros en order_products:", len(order_products))
print("Pedidos distintos en order_products:", order_products["order_id"].nunique())
print()
print("Pedidos de instacart_orders sin productos en order_products:",
      orders["order_id"].nunique() - order_products["order_id"].nunique())

Registros en instacart_orders: 478967
Pedidos distintos (order_id): 478952
Usuarios distintos (user_id): 157437

Registros en order_products: 4545007


Pedidos distintos en order_products: 450046

Pedidos de instacart_orders sin productos en order_products: 28906


En `instacart_orders` hay 478.967 registros, pero solo 478.952 pedidos distintos (los 15 duplicados) y 157.437 usuarios distintos. En `order_products` hay 4.545.007 registros (artículos) repartidos en 450.046 pedidos distintos. Eso deja 28.906 pedidos de `instacart_orders` sin productos registrados.

Cada usuario tiene en promedio unos 3 pedidos en este archivo, pero el promedio de `order_number` es 17. Esto sugiere que el archivo no incluye todo el historial de cada usuario, solo una parte de sus pedidos.

### Revisión adicional de `days_since_prior_order`

In [21]:
dias = orders["days_since_prior_order"]
print("Mediana:", dias.median())
print("Máximo:", dias.max())
print("Pedidos con exactamente 30 días:", (dias == 30).sum())

Mediana: 7.0
Máximo: 30.0
Pedidos con exactamente 30 días: 51338


La mediana es 7 días y el promedio es 11,1: algunos clientes tardan mucho más y suben el promedio. Además, el máximo es 30 y hay 51.338 pedidos con exactamente 30 días, lo que sugiere que el dato está topado en 30 (todo lo que pasa de ese plazo queda como 30). Eso hay que tenerlo presente al interpretar el promedio.

## Momento 6. Resolución del problema y presentación del producto

### Respuesta a la pregunta orientadora
Con Python y Pandas se puede cargar cada archivo en un DataFrame, revisar su estructura y sus valores ausentes, filtrar los registros que interesan y sacar indicadores con `mean()`, `sum()` y `describe()`. Con eso se identificó que los pedidos se concentran en el horario diurno, que los clientes vuelven a comprar con frecuencia y que la mayoría de los artículos comprados son recompras.

### Las tres preguntas de análisis y sus respuestas
1. **¿En qué horas del día se concentran los pedidos?** La mitad de los pedidos se hace entre las 10 y las 16 horas, la mediana es las 13 horas y hay pedidos a todas las horas (de 0 a 23).
2. **¿Cada cuánto tiempo suelen volver a comprar los clientes?** En promedio cada 11,1 días, con una mediana de 7 días. Además, 110.706 pedidos llegaron más de 15 días después del anterior.
3. **¿Qué parte de los productos que se compran son recompras?** 2.683.838 de 4.545.007 artículos, es decir, cerca del 59 %.

### Principales hallazgos
- **Pedidos:** hay 478.967 registros en `instacart_orders` (478.952 pedidos distintos) hechos por 157.437 usuarios distintos. El promedio de `order_number` es 17,16.
- **Productos:** el catálogo tiene 49.694 productos organizados en 134 pasillos y 21 departamentos. Los pedidos reúnen 4.545.007 artículos, y el pedido más grande tiene al menos 64.
- **Hábitos:** compras concentradas entre las 10 y las 16 horas, regreso frecuente y alta proporción de recompras.

### Cómo ayudaron la indexación y los filtros
Seleccionar columnas me permitió ver solo lo necesario (por ejemplo `product_id` y `product_name`). Los filtros con `loc[]` me permitieron aislar grupos concretos y contarlos: los 40.578 pedidos de las 10 horas, los 244.072 pedidos de clientes con más de 10 pedidos o los 2.683.838 artículos recomprados. Con dos condiciones se pueden hacer preguntas más precisas, como cuántos de los pedidos de las 10 horas son de clientes frecuentes (21.537).

### Limitaciones de los datos
- Hay valores ausentes en tres columnas: `days_since_prior_order` (28.819), `product_name` (1.258) y `add_to_cart_order` (836).
- Hay 15 pedidos duplicados en `instacart_orders`.
- 28.906 pedidos no tienen productos registrados en `order_products`.
- `days_since_prior_order` parece topado en 30 días, lo que afecta el promedio.
- El archivo parece tener solo una parte de los pedidos de cada usuario.
- No se aclara qué día de la semana es `order_dow` igual a 0.
- No hay datos de precios ni cantidades, así que no se puede analizar el dinero de las ventas.

### Conclusiones
1. **Los pedidos se concentran en horario diurno.** La mitad se hace entre las 10 y las 16 horas, lo que sugiere que la operación de Instacart debe estar más reforzada en ese horario.
2. **Los clientes recompran mucho y vuelven pronto.** Cerca del 59 % de los artículos son recompras y la mediana entre pedidos es de 7 días, lo que sugiere que funciones como "volver a comprar" pueden ser útiles para los clientes.
3. **Los datos sirven, pero hay que limpiarlos antes de un análisis más profundo.** Tienen nulos, duplicados y pedidos sin productos, y es importante diferenciar entre registros, pedidos y usuarios para no sacar conclusiones equivocadas.